In [7]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import json
import time

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
BASE_URL = "https://dps.psx.com.pk/company/"
HEADERS = {"User-Agent": "Mozilla/5.0"}

CSV_FILE = "company_name_value.csv"
MAX_COMPANIES = 1   # ONLY TEST FIRST 3

# ─────────────────────────────────────────────
# SCRAPE COMPANY PROFILE
# ─────────────────────────────────────────────
def scrape_company(symbol):
    url = f"{BASE_URL}{symbol}"
    print(f"Scraping PSX Profile: {symbol}")
    data = {"symbol": symbol}

    try:
        res = requests.get(url, headers=HEADERS, timeout=20)
        res.raise_for_status()
    except Exception as e:
        data["error"] = f"PSX Error: {str(e)}"
        return data

    soup = BeautifulSoup(res.text, "html.parser")

    # Business Description
    business_section = soup.find(
        "div",
        class_="item__head",
        string=lambda t: t and "BUSINESS DESCRIPTION" in t.upper()
    )
    if business_section:
        desc = []
        for sib in business_section.find_next_siblings():
            if sib.name == "div" and "item__head" in sib.get("class", []):
                break
            desc.append(sib.get_text(strip=True))
        data["business_description"] = " ".join(desc).strip()

    # Key People
    key_people_section = soup.find("div", class_="profile__item--people")
    if key_people_section:
        people = []
        for row in key_people_section.select("table.tbl tbody tr"):
            cols = row.find_all("td")
            if len(cols) >= 2:
                people.append({
                    "name": cols[0].get_text(strip=True),
                    "role": cols[1].get_text(strip=True)
                })
        data["key_people"] = people

    # Other Profile Fields
    for item in soup.select("div.profile__item"):
        for head in item.find_all("div", class_="item__head"):
            key = head.get_text(strip=True).lower().replace(" ", "_")
            val_tag = head.find_next_sibling(["p", "a"])
            if val_tag:
                data[key] = val_tag.get_text(strip=True)

    return data

# ─────────────────────────────────────────────
# SCRAPE EQUITY PROFILE
# ─────────────────────────────────────────────
def scrape_equity(symbol):
    url = f"{BASE_URL}{symbol}"
    print(f"Scraping Equity: {symbol}")
    data = {"symbol": symbol}

    try:
        res = requests.get(url, headers=HEADERS, timeout=20)
        res.raise_for_status()
    except Exception as e:
        data["error"] = f"Equity Error: {str(e)}"
        return data

    soup = BeautifulSoup(res.text, "html.parser")
    equity_section = soup.find("div", id="equity")

    if not equity_section:
        data["error"] = "Equity section not found"
        return data

    for item in equity_section.select("div.stats_item"):
        label_tag = item.find("div", class_="stats_label")
        value_tag = item.find("div", class_="stats_value")
        if label_tag and value_tag:
            label = (
                label_tag.get_text(strip=True)
                .lower()
                .replace(" ", "_")
                .replace("'", "")
            )
            data[label] = value_tag.get_text(strip=True)

    return data

# ─────────────────────────────────────────────
# SCRAPE LIVE QUOTE
# ─────────────────────────────────────────────
def scrape_quote(symbol):
    url = f"{BASE_URL}{symbol}"
    print(f"Scraping Quote: {symbol}")

    try:
        res = requests.get(url, headers=HEADERS, timeout=20)
        res.raise_for_status()
    except Exception as e:
        return {"error": f"Quote Error: {str(e)}"}

    soup = BeautifulSoup(res.text, "html.parser")

    name = soup.select_one(".quote__name")
    sector = soup.select_one(".quote__sector span")
    close_price = soup.select_one(".quote__close")
    change_value = soup.select_one(".change__value")
    change_percent = soup.select_one(".change__percent")

    stats = {
        item.select_one(".stats_label").text.strip():
        item.select_one(".stats_value").text.strip()
        for item in soup.select(".stats_item")
        if item.select_one(".stats_label") and item.select_one(".stats_value")
    }

    return {
        "company": name.text.strip() if name else "N/A",
        "sector": sector.text.strip() if sector else "N/A",
        "last_close": close_price.text.strip() if close_price else "N/A",
        "change": change_value.text.strip() if change_value else "N/A",
        "change_percent": change_percent.text.strip() if change_percent else "N/A",
        "open": stats.get("Open", "N/A"),
        "high": stats.get("High", "N/A"),
        "low": stats.get("Low", "N/A"),
        "volume": stats.get("Volume", "N/A"),
        "circuit_breaker": stats.get("CIRCUIT BREAKER", "N/A"),
        "day_range": stats.get("DAY RANGE", "N/A"),
        "week_52_range": stats.get("52-WEEK RANGE ^", "N/A"),
        "pe_ratio": stats.get("P/E Ratio (TTM) **", "N/A"),
        "one_year_change": stats.get("1-Year Change * ^", "N/A"),
        "ytd_change": stats.get("YTD Change * ^", "N/A"),
    }

# ─────────────────────────────────────────────
# FETCH FINANCIAL SUMMARY
# ─────────────────────────────────────────────
def fetch_financials(company_name, company_value):
    url = f"https://api.askanalyst.com.pk/api/companyfinancialnew/{company_value}?companyfinancial=true&test=true"
    rows = []

    try:
        response = requests.get(url, timeout=20)
        response.raise_for_status()
        data = response.json()

        for metric in data:
            label = metric.get("label")
            unit = metric.get("unit")

            for item in metric.get("data", []):
                try:
                    value = float(item.get("value"))
                except (TypeError, ValueError):
                    value = None

                rows.append({
                    "metric": label,
                    "unit": unit,
                    "year": str(item.get("year")),
                    "value": value
                })

        print(f"✓ Financials fetched: {company_name}")

    except Exception as e:
        print(f"✗ Financials failed: {company_name} → {e}")

    return rows

# ─────────────────────────────────────────────
# FETCH INDUSTRY
# ─────────────────────────────────────────────
def fetch_industry(company_name, company_value):
    url = f"https://api.askanalyst.com.pk/api/industrynew/{company_value}"
    print(f"Fetching Industry: {company_name}")
    try:
        res = requests.get(url, timeout=15)
        res.raise_for_status()
        return res.json()
    except Exception as e:
        return {"error": f"Industry Error: {str(e)}"}

# ─────────────────────────────────────────────
# FETCH STOCK PRICES
# ─────────────────────────────────────────────
def fetch_stock_price(company_name, company_value):
    url = f"https://api.askanalyst.com.pk/api/stockpricedatanew/{company_value}"
    print(f"Fetching Stock Prices: {company_name}")
    try:
        res = requests.get(url, timeout=15)
        res.raise_for_status()
        return res.json()
    except Exception as e:
        return {"error": f"Stock Price Error: {str(e)}"}

# ─────────────────────────────────────────────
# FETCH FINANCIAL STATEMENTS + RATIOS
# ─────────────────────────────────────────────
API_MAP = {
    "cash_flow": "https://api.askanalyst.com.pk/api/cf/{value}",
    "balance_sheet": "https://api.askanalyst.com.pk/api/bs/{value}",
    "income_statement": "https://api.askanalyst.com.pk/api/is/{value}",
    "financial_ratios": "https://api.askanalyst.com.pk/api/rationew/{value}",
}

def fetch_statements(company_name, company_value):
    print(f"Fetching Statements & Ratios: {company_name}")
    results = {}
    for key, url_tpl in API_MAP.items():
        url = url_tpl.format(value=company_value)
        try:
            res = requests.get(url, timeout=15)
            res.raise_for_status()
            results[key] = res.json()
        except Exception as e:
            results[key] = {"error": f"{key} Error: {str(e)}"}
    return results

# ─────────────────────────────────────────────
# MAIN TEST PIPELINE
# ─────────────────────────────────────────────
def main():
    df_companies = pd.read_csv(CSV_FILE).head(MAX_COMPANIES)
    final_output = []

    for _, row in df_companies.iterrows():
        name = row["name"]
        value = row["value"]
        symbol = row["symbol"]

        print(f"\nProcessing: {name}")

        company_block = {
            "name": name,
            "symbol": symbol,
            "financials": fetch_financials(name, value),
            "profile": scrape_company(symbol),
            "equity": scrape_equity(symbol),
            "quote": scrape_quote(symbol),
            "industry": fetch_industry(name, value),
            "stock_prices": fetch_stock_price(name, value),
            "statements": fetch_statements(name, value)
        }

        final_output.append(company_block)
        time.sleep(2)

    # PRINT JSON ONLY
    print("\n================ FINAL JSON OUTPUT ================\n")
    print(json.dumps(final_output, indent=2, ensure_ascii=False))

# ─────────────────────────────────────────────
if __name__ == "__main__":
    main()



Processing: Abbot Laboratories (Pakistan) Ltd
✓ Financials fetched: Abbot Laboratories (Pakistan) Ltd
Scraping PSX Profile: ABOT
Scraping Equity: ABOT
Scraping Quote: ABOT
Fetching Industry: Abbot Laboratories (Pakistan) Ltd
Fetching Stock Prices: Abbot Laboratories (Pakistan) Ltd
Fetching Statements & Ratios: Abbot Laboratories (Pakistan) Ltd

================ FINAL JSON OUTPUT ================

[
  {
    "name": "Abbot Laboratories (Pakistan) Ltd",
    "symbol": "ABOT",
    "financials": [
      {
        "metric": "EPS",
        "unit": "PKR",
        "year": "2013",
        "value": 25.83
      },
      {
        "metric": "EPS",
        "unit": "PKR",
        "year": "2014",
        "value": 28.77
      },
      {
        "metric": "EPS",
        "unit": "PKR",
        "year": "2015",
        "value": 36.64
      },
      {
        "metric": "EPS",
        "unit": "PKR",
        "year": "2016",
        "value": 41.08
      },
      {
        "metric": "EPS",
        "unit": "PKR"